# ARM97 Observation Variable Browser

This notebook visualizes one manually selected ARM97 IOP observation NetCDF file.

- Edit `OBSERVATION_FILE` in the setup cell, or paste a new file path into the widget.
- Surface variables are variables with `time` but no `lev` dimension; they are plotted as time series.
- Profile variables are variables with `time` and `lev`; the time-series view uses a pressure-level slider and plots one level at a time, while the heatmap view shows all levels.
- Pressure-coordinate heatmaps are shown in atmospheric orientation: low pressure/high altitude at the top and high pressure/near-surface levels at the bottom.


In [ ]:
from __future__ import annotations

from datetime import datetime, timedelta
import os
from pathlib import Path


def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "configs").exists() and (candidate / "notebooks").exists():
            return candidate
    return Path("/Users/yunlong/Workshop/SCM-UQ-Workflow")


ROOT = Path(os.environ.get("SCM_UQ_WORKFLOW_ROOT", find_repo_root())).resolve()
os.environ.setdefault("MPLCONFIGDIR", str(ROOT / ".local_cache/matplotlib-cache"))
os.environ.setdefault("XDG_CACHE_HOME", str(ROOT / ".local_cache"))

import numpy as np
import pandas as pd
from netCDF4 import Dataset
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display, clear_output

# Manually choose the observation file here. The widget below can also reload a different path.
OBSERVATION_FILE = Path(
    os.environ.get(
        "ARM97_OBSERVATION_FILE",
        str(ROOT / "outputs/arm97_iop_observation.nc"),
    )
).expanduser().resolve()

OUT_DIR = ROOT / "notebook_outputs" / "arm97_observation_all_variables"

assert ROOT.exists(), ROOT
assert OBSERVATION_FILE.exists(), OBSERVATION_FILE

print("repo root:", ROOT)
print("observation file:", OBSERVATION_FILE)
print("output dir:", OUT_DIR)


## Inspect Variables

In [ ]:
VERTICAL_DIMS = {"lev"}
COORDINATE_OR_TIME_VARS = {"time", "tsec", "bdate", "lat", "lon", "lev"}
BOOKKEEPING_TIME_VARS = {"year", "month", "day", "hour", "minute"}


def is_numeric_variable(var):
    return np.issubdtype(np.dtype(var.dtype), np.number)


def classify_variable(name, var):
    dims = tuple(var.dimensions)
    if "time" not in dims or not is_numeric_variable(var):
        return None
    if name in COORDINATE_OR_TIME_VARS or name in BOOKKEEPING_TIME_VARS:
        return None
    if "lev" in dims:
        return "profile"
    return "surface"


def variable_catalog(observation_file):
    rows = []
    with Dataset(observation_file) as ds:
        for name, var in ds.variables.items():
            category = classify_variable(name, var)
            if category is None:
                continue
            rows.append(
                {
                    "variable": name,
                    "category": category,
                    "dimensions": ", ".join(var.dimensions),
                    "shape": " x ".join(str(x) for x in var.shape),
                    "units": getattr(var, "units", ""),
                    "long_name": getattr(var, "long_name", ""),
                }
            )
    if not rows:
        return pd.DataFrame(columns=["variable", "category", "dimensions", "shape", "units", "long_name"])
    return pd.DataFrame(rows).sort_values(["category", "variable"]).reset_index(drop=True)


def load_catalog(observation_file):
    global OBSERVATION_FILE, CATALOG, SURFACE_CATALOG, PROFILE_CATALOG
    OBSERVATION_FILE = Path(observation_file).expanduser().resolve()
    if not OBSERVATION_FILE.exists():
        raise FileNotFoundError(OBSERVATION_FILE)
    CATALOG = variable_catalog(OBSERVATION_FILE)
    SURFACE_CATALOG = CATALOG[CATALOG["category"] == "surface"].reset_index(drop=True)
    PROFILE_CATALOG = CATALOG[CATALOG["category"] == "profile"].reset_index(drop=True)
    print(f"Loaded catalog from {OBSERVATION_FILE}")
    print(f"surface variables: {len(SURFACE_CATALOG)}")
    print(f"profile variables: {len(PROFILE_CATALOG)}")
    return CATALOG


catalog = load_catalog(OBSERVATION_FILE)
catalog


## Plot Helpers

In [ ]:
def filled(var_or_array):
    return np.asarray(np.ma.asarray(var_or_array, dtype=np.float64).filled(np.nan), dtype=np.float64)


def parse_bdate(ds):
    value = int(np.asarray(ds.variables["bdate"][...]).item())
    text = str(value)
    if len(text) == 8:
        return datetime(int(text[:4]), int(text[4:6]), int(text[6:8]))
    if len(text) == 6:
        year = int(text[:2])
        year += 1900 if year >= 70 else 2000
        return datetime(year, int(text[2:4]), int(text[4:6]))
    raise ValueError(f"unsupported bdate value: {value}")


def load_observation_time_axis(ds):
    if "bdate" in ds.variables and "tsec" in ds.variables:
        base = parse_bdate(ds)
        tsec = np.asarray(ds.variables["tsec"][:], dtype=np.float64)
        return np.array([base + timedelta(seconds=float(x)) for x in tsec], dtype=object)
    if "time" in ds.variables:
        return np.asarray(ds.variables["time"][:], dtype=np.float64)
    raise ValueError("observation file has no recognizable time axis")


def reduce_to_time_series(values, dims):
    values = filled(values)
    axes = tuple(i for i, dim in enumerate(dims) if dim != "time")
    if axes:
        return np.nanmean(values, axis=axes)
    return values


def reduce_to_time_level(values, dims):
    values = filled(values)
    time_axis = dims.index("time")
    level_axis = dims.index("lev")
    values = np.moveaxis(values, (time_axis, level_axis), (0, 1))
    if values.ndim > 2:
        values = np.nanmean(values, axis=tuple(range(2, values.ndim)))
    return values


def level_values(ds):
    values = filled(ds.variables["lev"][:]).squeeze()
    units = getattr(ds.variables["lev"], "units", "")
    if units.lower() == "pa":
        values = values / 100.0
        units = "hPa"
    label = f"lev ({units})" if units else "lev"
    return values, label


def sort_levels_low_to_high(levels, matrix):
    levels = np.asarray(levels, dtype=np.float64)
    order = np.argsort(levels)
    return levels[order], matrix[:, order]


def is_pressure_axis(level_label):
    lowered = level_label.lower()
    return "hpa" in lowered or "pa" in lowered or "pressure" in lowered


def color_range(values):
    finite = np.asarray(values)[np.isfinite(values)]
    if finite.size == 0:
        return None, None
    lo, hi = np.nanpercentile(finite, [2, 98])
    if np.isclose(lo, hi):
        return None, None
    return float(lo), float(hi)


def variable_title(name, var):
    long_name = getattr(var, "long_name", "")
    units = getattr(var, "units", "")
    text = f"<b>{name}</b>"
    if long_name:
        text += f": {long_name}"
    if units:
        text += f"<br><sup>Units: {units}</sup>"
    return text


def plot_surface_variable(variable_name):
    with Dataset(OBSERVATION_FILE) as ds:
        time_dates = load_observation_time_axis(ds)
        var = ds.variables[variable_name]
        dims = tuple(var.dimensions)
        units = getattr(var, "units", "")
        series = reduce_to_time_series(var[:], dims)
        fig = go.Figure(
            go.Scatter(
                x=time_dates,
                y=series,
                mode="lines",
                line=dict(color="#1261A6", width=2.0),
                hovertemplate="%{x}<br>value=%{y:.4g}<extra></extra>",
            )
        )
        subtitle = ""
        if any(dim != "time" and len(ds.dimensions[dim]) > 1 for dim in dims):
            subtitle = "<br><sup>Reduced by averaging non-time dimensions.</sup>"
        fig.update_layout(
            title=dict(text=variable_title(variable_name, var) + subtitle, x=0.01, xanchor="left", font=dict(size=22)),
            template="plotly_white",
            height=500,
            width=1120,
            hovermode="x unified",
            margin=dict(l=80, r=40, t=105, b=70),
        )
        fig.update_xaxes(title="Time")
        fig.update_yaxes(title=units or variable_name)
        return fig


def load_profile_matrix(variable_name):
    with Dataset(OBSERVATION_FILE) as ds:
        time_dates = load_observation_time_axis(ds)
        var = ds.variables[variable_name]
        dims = tuple(var.dimensions)
        matrix = reduce_to_time_level(var[:], dims)
        levels, level_label = level_values(ds)
        levels, matrix = sort_levels_low_to_high(levels, matrix)
        units = getattr(var, "units", "")
        title = variable_title(variable_name, var)
    return time_dates, levels, matrix, level_label, units, title


def profile_level_options(variable_name):
    _, levels, _, level_label, _, _ = load_profile_matrix(variable_name)
    options = [(f"{level:g}", int(idx)) for idx, level in enumerate(levels)]
    return options, level_label


def plot_profile_time_series(variable_name, level_index=0):
    time_dates, levels, matrix, level_label, units, title = load_profile_matrix(variable_name)
    level_index = int(np.clip(level_index, 0, len(levels) - 1))
    level = levels[level_index]
    series = matrix[:, level_index]

    fig = go.Figure(
        go.Scatter(
            x=time_dates,
            y=series,
            mode="lines",
            line=dict(color="#1261A6", width=2.0),
            hovertemplate=f"%{{x}}<br>{level_label}={level:g}<br>value=%{{y:.4g}}<extra></extra>",
        )
    )
    fig.update_layout(
        title=dict(
            text=title + f"<br><sup>{level_label}={level:g}; levels are sorted from low numeric level to high numeric level.</sup>",
            x=0.01,
            xanchor="left",
            font=dict(size=22),
        ),
        template="plotly_white",
        height=520,
        width=1120,
        hovermode="x unified",
        margin=dict(l=80, r=40, t=110, b=70),
    )
    fig.update_xaxes(title="Time")
    fig.update_yaxes(title=units or variable_name)
    return fig


def plot_profile_heatmap(variable_name):
    time_dates, levels, matrix, level_label, units, title = load_profile_matrix(variable_name)
    cmin, cmax = color_range(matrix)
    fig = go.Figure(
        go.Heatmap(
            x=time_dates,
            y=levels,
            z=matrix.T,
            colorscale="Viridis",
            zmin=cmin,
            zmax=cmax,
            colorbar=dict(title=units or variable_name),
            hovertemplate="%{x}<br>level=%{y}<br>value=%{z:.4g}<extra></extra>",
        )
    )
    fig.update_layout(
        title=dict(
            text=title + "<br><sup>Pressure-coordinate heatmaps use atmospheric orientation.</sup>",
            x=0.01,
            xanchor="left",
            font=dict(size=22),
        ),
        template="plotly_white",
        height=620,
        width=1120,
        margin=dict(l=80, r=80, t=115, b=70),
    )
    fig.update_xaxes(title="Time")
    fig.update_yaxes(title=level_label, autorange="reversed" if is_pressure_axis(level_label) else True)
    return fig


## Interactive Variable Browser

Paste or edit the observation file path, reload the catalog, then use the separate surface/profile controls below.


In [ ]:
observation_path = widgets.Text(
    value=str(OBSERVATION_FILE),
    description="Obs file",
    layout=widgets.Layout(width="980px"),
)
reload_button = widgets.Button(description="Load file", button_style="primary", icon="refresh")
load_output = widgets.Output()

surface_dropdown = widgets.Dropdown(description="Surface", layout=widgets.Layout(width="760px"))
surface_output = widgets.Output()

profile_dropdown = widgets.Dropdown(description="Profile", layout=widgets.Layout(width="760px"))
profile_plot_type = widgets.ToggleButtons(
    options=[("time series", "series"), ("heatmap", "heatmap")],
    value="series",
    description="Plot",
)
profile_level = widgets.SelectionSlider(
    options=[("loading", 0)],
    description="lev",
    continuous_update=False,
    readout=True,
    layout=widgets.Layout(width="620px"),
    style={"description_width": "45px"},
)
profile_output = widgets.Output()
_updating_profile_levels = False


def dropdown_options(df):
    options = []
    for _, row in df.sort_values("variable").iterrows():
        label = row["variable"]
        if row["long_name"]:
            label += f" | {row['long_name'][:80]}"
        options.append((label, row["variable"]))
    return options


def preferred_value(options, preferred):
    values = [value for _, value in options]
    for name in preferred:
        if name in values:
            return name
    return values[0] if values else None


def refresh_profile_levels(*_):
    global _updating_profile_levels
    if not profile_dropdown.value:
        profile_level.options = [("none", 0)]
        return
    _updating_profile_levels = True
    try:
        options, level_label = profile_level_options(profile_dropdown.value)
        profile_level.description = level_label.split()[0]
        profile_level.options = options
        if options:
            labels_as_float = np.array([float(label) for label, _ in options], dtype=float)
            default_idx = int(np.nanargmin(np.abs(labels_as_float - 500.0))) if np.nanmax(labels_as_float) > 100 else 0
            profile_level.value = options[default_idx][1]
    finally:
        _updating_profile_levels = False


def refresh_dropdowns():
    surface_options = dropdown_options(SURFACE_CATALOG)
    profile_options = dropdown_options(PROFILE_CATALOG)
    surface_dropdown.options = surface_options
    profile_dropdown.options = profile_options
    surface_default = preferred_value(surface_options, ["Tsair", "Tg", "Prec", "Ps"])
    profile_default = preferred_value(profile_options, ["T", "q", "u", "v", "omega", "rh"])
    if surface_default is not None:
        surface_dropdown.value = surface_default
    if profile_default is not None:
        profile_dropdown.value = profile_default
    refresh_profile_levels()


def reload_observation(_=None):
    with load_output:
        clear_output(wait=True)
        try:
            catalog = load_catalog(observation_path.value)
            display(catalog)
            refresh_dropdowns()
        except Exception as exc:
            print(f"Failed to load observation file: {exc!r}")


def redraw_surface(_=None):
    with surface_output:
        clear_output(wait=True)
        if not surface_dropdown.value:
            print("No surface variables found.")
            return
        display(plot_surface_variable(surface_dropdown.value))


def redraw_profile(_=None):
    if _updating_profile_levels:
        return
    with profile_output:
        clear_output(wait=True)
        if not profile_dropdown.value:
            print("No profile variables found.")
            return
        if profile_plot_type.value == "heatmap":
            profile_level.layout.display = "none"
            display(plot_profile_heatmap(profile_dropdown.value))
        else:
            profile_level.layout.display = None
            display(plot_profile_time_series(profile_dropdown.value, level_index=profile_level.value or 0))


def on_profile_variable_change(change=None):
    refresh_profile_levels()
    redraw_profile()


reload_button.on_click(reload_observation)
surface_dropdown.observe(redraw_surface, names="value")
profile_dropdown.observe(on_profile_variable_change, names="value")
profile_plot_type.observe(redraw_profile, names="value")
profile_level.observe(redraw_profile, names="value")

refresh_dropdowns()
display(widgets.VBox([widgets.HBox([observation_path, reload_button]), load_output]))
display(widgets.HTML("<h3>Surface observation variables: time series</h3>"))
display(widgets.VBox([surface_dropdown, surface_output]))
display(widgets.HTML("<h3>Profile observation variables: one-level time series or all-level heatmap</h3>"))
display(widgets.VBox([widgets.HBox([profile_dropdown, profile_plot_type]), profile_level, profile_output]))
redraw_surface()
redraw_profile()


## Save Variable Catalog

In [ ]:
OUT_DIR.mkdir(parents=True, exist_ok=True)
safe_name = "".join(ch if ch.isalnum() or ch in "._-" else "_" for ch in OBSERVATION_FILE.stem)
out = OUT_DIR / f"{safe_name}_variable_catalog.csv"
CATALOG.to_csv(out, index=False)
print("wrote", out)


## PDF Report

Run this section after loading the observation file and variable catalog. By default the report includes all available observation variables. The optional `REPORT_START_TIME` and `REPORT_END_TIME` settings crop every plot to the requested time window without changing the included variable count.


In [ ]:
from io import BytesIO
import textwrap

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm
from reportlab.lib import colors
from reportlab.lib.pagesizes import landscape, letter
from reportlab.lib.styles import getSampleStyleSheet
from reportlab.lib.units import inch
from reportlab.platypus import Image, PageBreak, Paragraph, SimpleDocTemplate, Spacer, Table, TableStyle


REPORT_SURFACE_VARS = None
REPORT_PROFILE_VARS = None

# Profile report pages use time-pressure heatmaps only.
REPORT_INCLUDE_PROFILE_HEATMAPS = True

# Optional PDF-only time crop. Use strings like "1997-06-25" or "1997-06-25 12:00".
# Leave as None to keep the full observation time range.

REPORT_START_TIME = None
REPORT_END_TIME = None

# REPORT_START_TIME = "1997-06-25"
# REPORT_END_TIME = "1997-07-05"


def _available_report_vars(requested, catalog_df):
    available = set(catalog_df["variable"])
    if requested is None:
        return [name for name in catalog_df["variable"]]
    return [name for name in requested if name in available]


def _describe_variable(variable_name):
    row = CATALOG[CATALOG["variable"] == variable_name]
    if row.empty:
        return "", ""
    first = row.iloc[0]
    return str(first.get("long_name", "")), str(first.get("units", ""))


def _time_window_label(start_time=None, end_time=None):
    start = "observation start" if start_time is None else str(start_time)
    end = "observation end" if end_time is None else str(end_time)
    return f"{start} to {end}"


def _time_window_mask(time_dates, start_time=None, end_time=None):
    stamps = pd.to_datetime([str(item) for item in time_dates], format="mixed")
    mask = np.ones(len(stamps), dtype=bool)
    if start_time is not None:
        mask &= stamps >= pd.Timestamp(start_time)
    if end_time is not None:
        mask &= stamps <= pd.Timestamp(end_time)
    if not mask.any():
        raise ValueError(f"time window has no samples: {_time_window_label(start_time, end_time)}")
    return mask


def _apply_time_window(time_dates, values, start_time=None, end_time=None):
    mask = _time_window_mask(time_dates, start_time=start_time, end_time=end_time)
    return np.asarray(time_dates, dtype=object)[mask], np.asarray(values)[mask]


def _plain_variable_title(variable_name):
    long_name, units = _describe_variable(variable_name)
    pieces = [variable_name]
    if long_name:
        pieces.append(long_name)
    if units:
        pieces.append(f"units: {units}")
    return " - ".join(pieces)


def _wrap_title(ax, title, width=92):
    ax.set_title("\n".join(textwrap.wrap(title, width=width)), loc="left", fontsize=12, pad=10)


def _png_from_figure(fig):
    buffer = BytesIO()
    fig.savefig(buffer, format="png", dpi=180, bbox_inches="tight")
    plt.close(fig)
    buffer.seek(0)
    return buffer


def _surface_report_png(variable_name, start_time=None, end_time=None):
    with Dataset(OBSERVATION_FILE) as ds:
        time_dates = load_observation_time_axis(ds)
        var = ds.variables[variable_name]
        dims = tuple(var.dimensions)
        units = getattr(var, "units", "")
        series = reduce_to_time_series(var[:], dims)
    time_dates, series = _apply_time_window(time_dates, series, start_time=start_time, end_time=end_time)

    fig, ax = plt.subplots(figsize=(10.8, 5.9))
    ax.plot(time_dates, series, color="#1261A6", linewidth=1.4)
    _wrap_title(ax, _plain_variable_title(variable_name))
    ax.set_xlabel("Time")
    ax.set_ylabel(units or variable_name)
    ax.grid(True, color="#d9d9d9", linewidth=0.6, alpha=0.8)
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%m-%d"))
    fig.autofmt_xdate()
    return _png_from_figure(fig)


def _nearest_level_indices(levels, requested_levels):
    levels = np.asarray(levels, dtype=np.float64)
    if requested_levels is None:
        return list(range(len(levels)))
    picked = []
    for target in requested_levels:
        idx = int(np.nanargmin(np.abs(levels - float(target))))
        if idx not in picked:
            picked.append(idx)
    return picked


def _profile_level_series_png(variable_name, requested_levels=None, start_time=None, end_time=None):
    time_dates, levels, matrix, level_label, units, _ = load_profile_matrix(variable_name)
    time_dates, matrix = _apply_time_window(time_dates, matrix, start_time=start_time, end_time=end_time)
    level_indices = _nearest_level_indices(levels, requested_levels)

    fig, ax = plt.subplots(figsize=(10.8, 5.9))
    colors_cycle = plt.cm.tab10(np.linspace(0, 1, max(len(level_indices), 1)))
    for color, idx in zip(colors_cycle, level_indices):
        ax.plot(time_dates, matrix[:, idx], linewidth=1.2, color=color, label=f"{levels[idx]:g} hPa")
    _wrap_title(ax, f"{_plain_variable_title(variable_name)} - selected pressure-level time series")
    ax.set_xlabel("Time")
    ax.set_ylabel(units or variable_name)
    ax.grid(True, color="#d9d9d9", linewidth=0.6, alpha=0.8)
    ax.legend(title=level_label, ncols=min(4, max(len(level_indices), 1)), fontsize=8, title_fontsize=8, loc="best")
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%m-%d"))
    fig.autofmt_xdate()
    return _png_from_figure(fig)


def _profile_heatmap_png(variable_name, start_time=None, end_time=None):
    time_dates, levels, matrix, level_label, units, _ = load_profile_matrix(variable_name)
    time_dates, matrix = _apply_time_window(time_dates, matrix, start_time=start_time, end_time=end_time)
    vmin, vmax = color_range(matrix)

    fig, ax = plt.subplots(figsize=(10.8, 5.9))
    mesh = ax.pcolormesh(time_dates, levels, matrix.T, shading="auto", cmap="viridis", vmin=vmin, vmax=vmax)
    if is_pressure_axis(level_label):
        ax.invert_yaxis()
    _wrap_title(ax, f"{_plain_variable_title(variable_name)} - time-pressure heatmap")
    ax.set_xlabel("Time")
    ax.set_ylabel(level_label)
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%m-%d"))
    fig.autofmt_xdate()
    cbar = fig.colorbar(mesh, ax=ax, pad=0.02)
    cbar.set_label(units or variable_name)
    return _png_from_figure(fig)


def _add_header(story, title, subtitle=None):
    styles = getSampleStyleSheet()
    story.append(Paragraph(title, styles["Title"]))
    if subtitle:
        story.append(Paragraph(subtitle, styles["BodyText"]))
    story.append(Spacer(1, 0.18 * inch))


def _add_png_page(story, title, png_buffer):
    _add_header(story, title)
    img = Image(png_buffer, width=9.8 * inch, height=5.35 * inch)
    story.append(img)
    story.append(PageBreak())


def _summary_table(story, surface_vars, profile_vars, start_time=None, end_time=None):
    styles = getSampleStyleSheet()
    _add_header(
        story,
        "ARM97 Observation Variables Report",
        f"Observation file: {OBSERVATION_FILE}<br/>Time window: {_time_window_label(start_time, end_time)}",
    )
    rows = [
        ["Item", "Value"],
        ["Surface variables", str(len(surface_vars))],
        ["Profile variables", str(len(profile_vars))],
        ["Profile pages", "time-pressure heatmap only"],
    ]
    table = Table(rows, colWidths=[2.7 * inch, 7.1 * inch])
    table.setStyle(
        TableStyle(
            [
                ("BACKGROUND", (0, 0), (-1, 0), colors.HexColor("#1f4e79")),
                ("TEXTCOLOR", (0, 0), (-1, 0), colors.white),
                ("FONTNAME", (0, 0), (-1, 0), "Helvetica-Bold"),
                ("GRID", (0, 0), (-1, -1), 0.35, colors.HexColor("#c8c8c8")),
                ("VALIGN", (0, 0), (-1, -1), "TOP"),
                ("ROWBACKGROUNDS", (0, 1), (-1, -1), [colors.white, colors.HexColor("#f5f7fa")]),
            ]
        )
    )
    story.append(table)
    story.append(Spacer(1, 0.24 * inch))

    variable_style = styles["BodyText"]
    variable_style.fontSize = 8
    variable_style.leading = 10
    preview_rows = [["Category", "Variables"]]
    preview_rows.append(["Surface", Paragraph(", ".join(surface_vars), variable_style)])
    preview_rows.append(["Profile", Paragraph(", ".join(profile_vars), variable_style)])
    preview = Table(preview_rows, colWidths=[1.3 * inch, 8.5 * inch])
    preview.setStyle(
        TableStyle(
            [
                ("BACKGROUND", (0, 0), (-1, 0), colors.HexColor("#5b6770")),
                ("TEXTCOLOR", (0, 0), (-1, 0), colors.white),
                ("FONTNAME", (0, 0), (-1, 0), "Helvetica-Bold"),
                ("GRID", (0, 0), (-1, -1), 0.35, colors.HexColor("#c8c8c8")),
                ("VALIGN", (0, 0), (-1, -1), "TOP"),
            ]
        )
    )
    story.append(preview)
    story.append(PageBreak())


def build_observation_pdf_report(
    output_pdf=None,
    surface_vars=None,
    profile_vars=None,
    start_time=None,
    end_time=None,
):
    surface_vars = _available_report_vars(surface_vars if surface_vars is not None else REPORT_SURFACE_VARS, SURFACE_CATALOG)
    profile_vars = _available_report_vars(profile_vars if profile_vars is not None else REPORT_PROFILE_VARS, PROFILE_CATALOG)
    safe_name = "".join(ch if ch.isalnum() or ch in "._-" else "_" for ch in OBSERVATION_FILE.stem)
    if output_pdf is None:
        OUT_DIR.mkdir(parents=True, exist_ok=True)
        output_pdf = OUT_DIR / f"{safe_name}_observation_variables_report.pdf"
    output_pdf = Path(output_pdf).expanduser().resolve()
    output_pdf.parent.mkdir(parents=True, exist_ok=True)

    story = []
    _summary_table(story, surface_vars, profile_vars, start_time=start_time, end_time=end_time)

    for variable_name in surface_vars:
        png = _surface_report_png(variable_name, start_time=start_time, end_time=end_time)
        _add_png_page(story, f"Surface: {variable_name}", png)

    for variable_name in profile_vars:
        png = _profile_heatmap_png(variable_name, start_time=start_time, end_time=end_time)
        _add_png_page(story, f"Profile: {variable_name} heatmap", png)

    if story and isinstance(story[-1], PageBreak):
        story.pop()

    doc = SimpleDocTemplate(
        str(output_pdf),
        pagesize=landscape(letter),
        rightMargin=0.45 * inch,
        leftMargin=0.45 * inch,
        topMargin=0.38 * inch,
        bottomMargin=0.38 * inch,
    )
    doc.build(story)
    print(f"wrote {output_pdf}")
    print(f"pages: {1 + len(surface_vars) + len(profile_vars)}")
    return output_pdf


report_pdf = build_observation_pdf_report(start_time=REPORT_START_TIME, end_time=REPORT_END_TIME)
report_pdf
